In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()


# ============================================================
# 1. CV
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_lof_score = np.zeros(
    len(X_array)
)


# ============================================================
# 2. OOF LOF
# ============================================================

for fold, (
    train_idx,
    val_idx
) in enumerate(
    cv.split(
        X_array,
        y_array
    ),
    start=1
):

    print(
        f"Fold {fold}"
    )

    X_train = X_array[
        train_idx
    ]

    y_train = y_array[
        train_idx
    ]

    X_val = X_array[
        val_idx
    ]


    # 정상 데이터만 학습
    X_train_normal = X_train[
        y_train == 0
    ]


    model = LocalOutlierFactor(
        n_neighbors=20,
        novelty=True,
        contamination="auto",
        n_jobs=-1
    )

    model.fit(
        X_train_normal
    )


    # decision_function:
    # 정상일수록 큰 값
    # 이상일수록 작은 값
    #
    # 부호 반전 → 클수록 이상
    anomaly_score = -model.decision_function(
        X_val
    )


    oof_lof_score[
        val_idx
    ] = anomaly_score


# ============================================================
# 3. PR-AUC
# ============================================================

pr_auc = average_precision_score(
    y_array,
    oof_lof_score
)

print(
    "\nLOF OOF PR-AUC:",
    pr_auc
)


# ============================================================
# 4. 전체 데이터 Rank
# ============================================================

result_df = pd.DataFrame({

    "Target":
        y_array,

    "LOF_Score":
        oof_lof_score
})


result_df[
    "LOF_Rank"
] = result_df[
    "LOF_Score"
].rank(
    ascending=False,
    method="min"
)


# ============================================================
# 5. 불량 Rank
# ============================================================

defect_rank_df = (
    result_df[
        result_df["Target"] == 1
    ]
    .sort_values(
        "LOF_Rank"
    )
)


print(
    "\n[Defect LOF Rank]"
)

display(
    defect_rank_df
)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

LOF OOF PR-AUC: 0.2096689205407975

[Defect LOF Rank]


,Target,LOF_Score,LOF_Rank
118,1,2.676686,5.0
119,1,2.652686,6.0
120,1,2.598677,7.0
117,1,2.508646,8.0
115,1,2.477107,9.0
116,1,2.356037,10.0
95,1,0.077886,20.0
107,1,-0.199524,34.0
113,1,-0.343964,172.0
111,1,-0.345478,175.0
